# 02 ROS 入门与话题通信

> 覆盖范围：ROS 核心概念（节点/话题/服务/动作的**原理与选型**）、工作空间与构建（catkin/colcon）、消息（msg/srv/action）与代码生成、话题通信手写（发布/订阅/一对多/异步/服务/动作全部可跑）、TF 坐标变换、URDF 建模、launch 启动、仿真（Gazebo）、ROS1 vs ROS2。
> 使用方式：先读**原理图景**（为什么这样设计），再看 **rospy / rclpy 真实 API**（阅读与默写），最后用手写的最小实现**验证语义**，自测面试追问。

---

## 1. ROS 是什么：不是操作系统，是机器人软件框架

ROS（Robot Operating System）**不是操作系统**，而是一套运行在 Linux（Ubuntu 主流）之上的**机器人软件框架**，解决机器人开发的三个基本问题：

1. **进程间通信**：摄像头驱动、激光驱动、定位、规划、控制……几十个程序如何高效互传数据？
2. **模块复用**：A 团队写的导航算法能否直接接到 B 团队的底盘上？→ 靠**标准消息接口**；
3. **工具链**：可视化（rviz）、仿真（Gazebo）、坐标变换（TF）、机器人建模（URDF）、批量启动（launch）。

### 1.1 核心抽象：节点 + 话题 + 服务 + 动作

ROS 把整个机器人系统看作一张**计算图（computational graph）**：

- **节点（Node）**：最小的可执行单元（一个进程或进程内的一个模块），如 `/camera_driver`、`/lidar_driver`、`/planner`、`/controller`；
- **话题（Topic）**：**异步发布/订阅**通道，数据单向流动、一对多，适合**高频传感器数据流**（图像 30Hz、激光 10Hz）；
- **服务（Service）**：**同步请求/响应**，一问一答，适合**低频命令**（「打开机械臂」、「查询地图」);
- **动作（Action）**：**目标/反馈/结果**三段式，适合**长时任务**（「导航到坐标点」，执行中持续回报进度、可随时取消）；
- **参数（Parameter）**：全局键值对，供节点读取配置。

| 原语 | 模式 | 同步/异步 | 方向 | 类比 | 典型场景 |
| --- | --- | --- | --- | --- | --- |
| 话题 Topic | 发布/订阅 | 异步 | 单向一对多 | 广播电台 | 图像/激光/里程计/速度指令 |
| 服务 Service | 请求/响应 | 同步 | 一问一答 | 电话问答 | 开关、查询、短暂命令 |
| 动作 Action | 目标/反馈/结果 | 异步+进度 | 双向长任务 | 快递派单跟踪 | 导航、机械臂轨迹、建图 |

> **选型一句话**：数据流用 Topic；一次性命令用 Service；长任务要进度/取消用 Action。

## 2. 节点与图：ROS1 的 roscore 与 ROS2 的 DDS

### 2.1 节点如何互相发现

- **ROS1**：节点启动时向**中心节点 roscore** 注册（名字、话题、服务）。发布者告知 roscore「我在发 /scan」，订阅者查询「谁在发 /scan」→ roscore 牵线，双方建立**点对点 TCPROS** 连接，之后数据不再经过 roscore。roscore 是**单点故障**：它挂了整个系统瘫痪；
- **ROS2**：基于 **DDS（Data Distribution Service，数据分发服务）**，采用**去中心化的自动发现**：节点通过 UDP 组播互相发现并协商 QoS，**不需要中心节点**，天然适合多机器人/分布式/强实时场景。

## 3. 工作空间与构建：catkin vs colcon

### 3.1 ROS1（catkin）

```bash
mkdir -p ~/catkin_ws/src && cd ~/catkin_ws/src
catkin_init_workspace          # 生成 CMakeLists.txt 符号链接
cd ~/catkin_ws && catkin_make  # 构建整个工作空间
source devel/setup.bash        # 把构建产物加入环境（每个新终端都要 source）
```

### 3.2 ROS2（colcon）

```bash
mkdir -p ~/ros2_ws/src && cd ~/ros2_ws
colcon build                    # 构建所有包（支持增量构建、--packages-select 选包）
source install/setup.bash
```

**包（package）**是 ROS 的基本组织单元：一个包 = 一个可编译单元（`package.xml` + `CMakeLists.txt` + 源码 + 接口定义）。

## 4. 消息与接口定义：msg / srv / action

ROS 的接口用**独立文件**定义，构建时**自动生成**各语言代码（Python/ C++/…），保证跨语言一致：

```text
msg/Demo.msg        # 话题消息
float32 x
float32 y
string label
```

```text
srv/AddTwoInts.srv  # 服务：request ---  --- response
int64 a
int64 b
---
int64 sum
```

常用标准消息：`std_msgs/String`（data）、`geometry_msgs/Pose`（位置+姿态）、`geometry_msgs/Twist`（线速度+角速度）、`sensor_msgs/Image`（图像）、`sensor_msgs/LaserScan`（激光）。

> **面试细节**：`header`（时间戳 + frame_id）几乎每个传感器消息都带——靠它做**数据同步与坐标关联**（哪个时刻、哪个坐标系下的数据）。


## 核心概念

### (a) 🗂️ ROS 的节点、通信原语与工具链主线

这篇讲 ROS 的完整图景：它不是操作系统，而是机器人软件框架；核心是「节点 + 话题/服务/动作」三类通信原语，再加上 TF/URDF/launch 工具链、QoS 与 ROS1 vs ROS2 的演进，学完就能看懂机器人系统的数据流。

```mermaid
flowchart TB
    Node1["🧠 ROS：机器人软件框架"] --> Node2["🎯 核心抽象：节点 / 话题 / 服务 / 动作"]
    Node2 --> Node3["📡 话题 Topic：异步一对多"]
    Node2 --> Node4["☎️ 服务 Service：同步一问一答"]
    Node2 --> Node5["📦 动作 Action：目标 / 反馈 / 结果"]
    Node3 --> Node6["🔧 手写发布/订阅：消息总线 + 回调"]
    Node1 --> Node7["🗺️ TF 坐标变换 + URDF 建模"]
    Node1 --> Node8["🚀 launch 启动 + rviz / Gazebo"]
    Node8 --> Node9["📶 QoS 与 ROS1 vs ROS2"]
```

- **🧠 ROS：机器人软件框架**：ROS（Robot Operating System）不是操作系统，是跑在 Linux 上的机器人软件框架，解决「几十个进程怎么互传数据、模块怎么复用、怎么可视化调试」这三大问题。
- **🎯 核心抽象**：整个系统是一张计算图：节点是最小可执行单元（一个进程，如摄像头驱动、规划器）；话题/服务/动作是三种通信原语；参数是全局键值配置。选型口诀：数据流用 Topic，一次性命令用 Service，长任务要进度用 Action。
- **📡 话题 Topic**：异步发布/订阅、单向一对多，像广播电台——发布者只管发，订阅者注册回调收。适合图像、激光这类高频传感器数据流（30Hz/10Hz）。
- **☎️ 服务 Service**：同步请求/响应、一问一答，像打电话：客户端发请求、服务端处理完返回结果。适合「打开机械臂」「查询地图」这类低频命令，不能用于高频数据流。
- **📦 动作 Action**：目标/反馈/结果三段式，像快递派单跟踪：下发目标、执行中持续回报进度、可随时取消。适合导航、机械臂轨迹这类长时任务。
- **🔧 手写发布/订阅**：用纯 Python 模拟话题总线语义：`publish(msg)` 把消息分发给所有已订阅的回调；用独立线程模拟异步节点（类似 `rospy.spin` 事件循环）；`queue_size` 用来缓冲慢消费者。
- **🗺️ TF 坐标变换 + URDF 建模**：机器人有多个坐标系（map/odom/base_link/laser/camera），TF 维护它们之间的变换、构成一棵树，任何带 `frame_id` 的传感器数据都能变换到任意坐标系；URDF 用 link（刚体）+ joint（关节）描述机器人外形。
- **🚀 launch 启动 + rviz / Gazebo**：launch 文件一键启动整机所有节点（ROS1 用 XML、ROS2 用 Python）；rviz 负责「看数据」不仿真物理，Gazebo 是物理仿真器（动力学、碰撞、传感器），典型流程是 Gazebo 仿真 + rviz 可视化。
- **📶 QoS 与 ROS1 vs ROS2**：QoS 是通信质量约定：高频传感器用 Best-Effort（丢帧无所谓、低延迟），控制指令/地图用 Reliable（不能丢），发布与订阅不兼容就「连不上」且不报错。ROS1 靠中心节点 roscore 牵线（单点故障），ROS2 用 DDS 去中心化自动发现，天然适合多机器人。


## 5. ROS1 发布/订阅（rospy 真实 API）

```python
# 发布者（talker）
import rospy
from std_msgs.msg import String

rospy.init_node('talker')                       # 注册节点
pub = rospy.Publisher('chatter', String, queue_size=10)  # 声明话题 + 类型 + 缓冲
rate = rospy.Rate(10)                           # 10 Hz 定时
while not rospy.is_shutdown():
    pub.publish(String('hello'))                # 构造消息并发布
    rate.sleep()
```

```python
# 订阅者（listener）
import rospy
from std_msgs.msg import String

def cb(msg):                                    # 回调：数据到达即被调用
    rospy.loginfo('收到: %s', msg.data)

rospy.init_node('listener')
rospy.Subscriber('chatter', String, cb)         # 注册订阅 + 回调
rospy.spin()                                    # 阻塞，持续处理回调队列
```

> 调试命令：`rostopic list` / `rostopic echo /chatter` / `rostopic hz /chatter` / `rostopic pub /chatter std_msgs/String 'data: hi'` / `rqt_graph`（可视化节点图）。

## 6. ROS2 发布/订阅（rclpy 真实 API）

```python
import rclpy
from rclpy.node import Node
from std_msgs.msg import String

class Talker(Node):                             # 继承 Node，节点即类
    def __init__(self):
        super().__init__('talker')
        self.pub = self.create_publisher(String, 'chatter', 10)
        self.timer = self.create_timer(0.1, self.tick)   # 100ms 定时回调

    def tick(self):
        self.pub.publish(String(data='hello'))

rclpy.init()
node = Talker()
rclpy.spin(node)                                # 事件循环
rclpy.shutdown()
```

> ROS2 命令：`ros2 node list` / `ros2 topic list` / `ros2 topic echo /chatter` / `ros2 topic info /chatter` / `ros2 run <pkg> <node>` / `ros2 launch <pkg> <launch>`。

**对比小结**：rospy 是「过程式 + 全局状态」，rclpy 是「面向对象 + 节点即类实例」；ROS2 的 QoS 设置、生命周期、DDS 中间件在 API 层直接可见。


## 7. 手写最小发布/订阅（Python 模拟真实语义）

下面用纯 Python 模拟 ROS 话题通信的核心语义——**发布-订阅、回调分发、一对多、异步节点、服务请求/响应、动作反馈**，在任何环境都能执行验证。真实环境请用上文 `rospy` / `rclpy` API。

### 7.1 话题总线：一对多分发

话题的底层模型：发布者调用 `publish(msg)` → 话题对象把它**分发给所有已订阅的回调**。队列（queue_size）是为了**缓冲慢消费者**——发布快于处理时先排队，满了再丢最老（ROS2 QoS 控制这一点）。


In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
import threading, time

class MiniTopic:
    """最小话题总线：模拟 ROS Topic 发布/订阅 + 一对多分发"""
    def __init__(self, name, queue_size=10):
        self.name = name
        self.queue_size = queue_size
        self._subs = []          # 订阅者回调列表
        self._queue = []         # 模拟队列缓冲

    def advertise(self):
        return Publisher(self)

    def subscribe(self, callback):
        self._subs.append(callback)
        return Subscriber(self, callback)

    def publish_msg(self, msg):
        # 队列缓冲：满了丢最老（近似 ROS queue_size 行为）
        self._queue.append(msg)
        if len(self._queue) > self.queue_size:
            self._queue.pop(0)
        for cb in list(self._subs):      # 一对多分发
            cb(msg)

class Publisher:
    def __init__(self, topic):
        self.topic = topic
    def publish(self, msg):
        print(f'[pub]   {self.topic.name}: {msg}')
        self.topic.publish_msg(msg)

class Subscriber:
    def __init__(self, topic, callback):
        self.topic, self.callback = topic, callback

def make_msg(type_name, **fields):
    """模拟 ROS msg 对象（如 std_msgs/String: data 字段），支持 data=... 实例化"""
    def _init(self, **kw):
        self.__dict__.update(kw)
    return type(type_name, (), dict(fields, __init__=_init))

# 建话题
chatter = MiniTopic('/chatter')

# 两个订阅者（一对多）
got = []
chatter.subscribe(lambda m: got.append(('A', m.data)))
chatter.subscribe(lambda m: got.append(('B', m.data)))

# 发布者以 10Hz 发 3 条
pub = chatter.advertise()
String = make_msg('String')
for i in range(3):
    pub.publish(String(data=f'hello-{i}'))
    time.sleep(0.1)

print('\n两个订阅者各收到 3 条:', got)
assert len(got) == 6, '一对多分发应收到 6 条'

In [ ]:
# 用独立线程模拟异步节点：talker 线程发布，listener 线程回调（类似 rospy.spin 事件循环）
import threading, time

class MiniNode:
    """模拟一个 ROS 节点：持有话题句柄，可定时发布"""
    _global_topics = {}  # 全局话题总线：同名话题全节点共享（真实 ROS 由 DDS 全局寻址）

    def __init__(self, name):
        self.name = name
        self.topics = {}

    def _topic(self, name):
        # 同一话题名全局共享同一个 MiniTopic，实现"跨节点"订阅送达
        if name not in MiniNode._global_topics:
            MiniNode._global_topics[name] = MiniTopic(name)
        return MiniNode._global_topics[name]

    def create_publisher(self, topic, msg_type):
        t = self._topic(topic)
        self.topics[topic] = t
        return t.advertise()

    def create_subscriber(self, topic, msg_type, callback):
        t = self._topic(topic)
        self.topics[topic] = t
        return t.subscribe(callback)

    def spin(self, seconds):
        time.sleep(seconds)

talker = MiniNode('talker')
listener = MiniNode('listener')
Shared = make_msg('String')

pub = talker.create_publisher('/chatter', Shared)
recv = []
listener.create_subscriber('/chatter', Shared, lambda m: recv.append(m.data))

# 单线程演示（真实场景里 publish 与回调在不同节点/线程）
for i in range(5):
    pub.publish(Shared(data=f'msg-{i}'))
print('listener 收到:', recv)
assert recv == [f'msg-{i}' for i in range(5)], '话题消息应全部送达'


# 真实线程版：talker 在子线程以固定频率发布
recv2 = []
t2 = MiniTopic('/scan')
p2 = t2.advertise()
t2.subscribe(lambda m: recv2.append(m.angle))

def publisher_loop():
    for i in range(4):
        p2.publish(Shared(angle=i * 10.0))   # 模拟 sensor_msgs/LaserScan 角度
        time.sleep(0.02)

th = threading.Thread(target=publisher_loop)
th.start(); th.join()
print('子线程发布4条, 订阅收到:', recv2)
assert len(recv2) == 4

In [ ]:
# Service 服务通信模拟（同步请求/响应）——类似 AddTwoInts 服务
def make_service(name, handler):
    def call(request):
        return handler(request)
    return {'name': name, 'call': call}

def add_two_ints(req):
    return {'sum': req['a'] + req['b']}   # 模拟 srv: int64 a, int64 b -> int64 sum

srv = make_service('/add_two_ints', add_two_ints)
for req in ({'a': 1, 'b': 2}, {'a': 10, 'b': -3}):
    resp = srv['call'](req)
    print(f"请求 {req} -> 响应 {resp}")
    assert resp['sum'] == req['a'] + req['b']


# Action 动作通信模拟（目标/反馈/结果三段式）——类似 Nav 导航动作
def run_action(goal, on_feedback, steps=5):
    """模拟动作服务器：收到目标 -> 逐步反馈进度 -> 返回结果"""
    print(f'[action server] 收到目标: {goal}')
    for i in range(1, steps + 1):
        on_feedback(f'{i}/{steps}')      # 持续反馈
    return {'success': True, 'result': f'完成 {goal}'}

feedback_log = []
result = run_action('移动到 (1,2)', feedback_log.append)
print('反馈序列:', feedback_log)
print('最终结果:', result['result'])
assert len(feedback_log) == 5 and result['success']

## 8. TF 坐标变换与 URDF 建模

### 8.1 TF：机器人的坐标系树

一个机器人有多个坐标系：`map`（地图）、`odom`（里程计）、`base_link`（底盘）、`laser`（激光）、`camera`（相机）。**TF 维护这些坐标系之间的变换关系**（平移+旋转，构成一棵树，根通常是 `map` 或 `odom`）：

- 任何传感器数据都带着 `frame_id`（如 `laser`），通过 TF 可变换到任意坐标系（如把激光点转到 `base_link`）；
- 广播端：`TransformBroadcaster`（如 `robot_state_publisher` 从 URDF 发布关节变换）；
- 查询端：`Buffer` + `TransformListener`，`lookup_transform(target, source, time)` 得到两个坐标系间的变换；
- TF 是**树状**结构：两坐标系间**只有一条路径**，否则 TF 会报「frame not found / 多重父节点」错误；
- 还支持**时间插值**：查询历史时刻的变换（数据同步时常用）。

```python
# ROS2 tf2 广播/查询示例（阅读）
# from tf2_ros import TransformBroadcaster, Buffer, TransformListener
# broadcaster.sendTransform(transform_stamped)
# trans = buffer.lookup_transform('base_link', 'laser', rclpy.time.Time())
# trans.transform.translation.x/y/z + trans.transform.rotation.x/y/z/w
```

### 8.2 URDF：连杆（link）+ 关节（joint）

URDF 是 XML 格式的机器人描述：**link** 是刚体（含惯性、视觉、碰撞几何），**joint** 连接两个 link 并定义运动类型。

```xml
<robot name="my_robot">
  <link name="base_link">
    <visual><geometry><box size="0.4 0.2 0.1"/></geometry></visual>
  </link>
  <joint name="base_to_wheel" type="continuous">
    <parent link="base_link"/>
    <child link="wheel"/>
    <axis xyz="0 1 0"/>
  </joint>
</robot>
```

**joint 类型**：`revolute`（旋转、有角度限制）、`continuous`（连续旋转，如轮子）、`prismatic`（平移）、`fixed`（固定）。

### 8.3 launch：一键启动整机

```xml
<!-- ROS1 launch -->
<launch>
  <node pkg="turtlesim" type="turtlesim_node" name="turtle1" output="screen"/>
  <node pkg="my_robot" type="controller" name="ctrl"/>
</launch>
```

```python
# ROS2 Python launch
from launch import LaunchDescription
from launch_ros.actions import Node

def generate_launch_description():
    return LaunchDescription([
        Node(package='turtlesim', executable='turtlesim_node', name='turtle1'),
        Node(package='my_robot', executable='controller', name='ctrl')
    ])
```

### 8.4 rviz vs Gazebo

- **rviz**：**可视化**工具——显示传感器数据（点云/图像/TF/路径）、调试话题，**不仿真物理**；
- **Gazebo**：**物理仿真器**——动力学、碰撞、摩擦、传感器仿真，跑真实控制算法；
- 典型流程：Gazebo 里仿真 + rviz 里看数据 + rqt_graph 看节点图。


## 9. QoS（ROS2）：通信质量的约定

ROS2 每个话题/服务都可以协商 **QoS（Quality of Service）** 策略，发布与订阅必须**兼容**：

| 策略 | 选项 | 含义 |
| --- | --- | --- |
| Reliability | Reliable / Best-Effort | 可靠传输（重发、确认）vs 尽力而为（丢包不重发，延迟低） |
| Durability | Volatile / Transient Local | 数据是否对后加入的订阅者保留（如地图需要保留） |
| History/Depth | Keep Last(N) / Keep All | 队列缓存策略 |
| Deadline | 时长 | 数据到达的最长间隔（超时判定） |

**选型经验**：

- 激光/相机这类**高频传感器**常用 `Best-Effort`（丢一帧无所谓，低延迟更重要）；
- **控制指令 / 地图 / 任务**用 `Reliable`（不能丢）；
- 订阅者 QoS 与发布者不兼容 → 话题「连不上」但节点不报错，调试时先查 `ros2 topic info` 的 QoS 一致性。

## 10. ROS1 vs ROS2 总对比

| 维度 | ROS1 | ROS2 |
| --- | --- | --- |
| 通信中间件 | 自定义 TCPROS/UDPROS | **DDS**（去中心化、QoS 可配置） |
| 主节点 | 需要 `roscore`（单点故障） | 不需要（DDS 自动发现） |
| 构建工具 | catkin / catkin_make | **colcon** |
| 语言 API | rospy / roscpp（过程式） | rclpy / rclcpp（节点即类） |
| 接口定义 | msg / srv（genmsg） | msg / srv / action + **IDL** |
| 生命周期 | 普通进程 | 生命周期节点（Unconfigured/Inactive/Active） |
| 多机器人 | 弱（全局命名空间冲突） | 强（domain_id + namespace 隔离） |
| 实时性/安全 | 弱（无 QoS、无加密） | 强（QoS、DDS 安全，可接工业/车规） |
| 生态现状 | 成熟（TurtleBot2、旧教材多） | 主流（2022+ 新项目默认 ROS2） |

## 11. 自测清单（面试追问）

- [ ] Topic / Service / Action 的区别？分别适合什么场景？一句话选型口诀？
- [ ] 为什么 Topic 是异步一对多？Service 为什么不能用于高频数据流？（同步阻塞、一次一问）
- [ ] ROS1 为什么要 roscore？它挂了会怎样？ROS2 怎么解决？（DDS 自动发现、去中心化）
- [ ] 手写：`rospy` 发布与订阅各 5 行；`rclpy` 继承 Node 创建定时发布。
- [ ] 自定义 msg 的文件结构与构建流程？`header` 字段为什么重要？
- [ ] TF 作用？为什么必须是树结构？`lookup_transform` 参数是什么？
- [ ] URDF 里 link 与 joint 的区别？joint 有哪几种类型？
- [ ] QoS（ROS2）是什么？Best-Effort vs Reliable 分别用于哪些话题？不兼容会怎样？
- [ ] 如何调试话题：`rostopic list/echo/hz`；`ros2 topic list/echo/hz/info`；节点图怎么画？
- [ ] 「订阅相机图像、发布速度指令」的节点图怎么画？（camera->vision->planner->controller）
- [ ] rviz 与 Gazebo 的区别？（可视化 vs 物理仿真）
- [ ] 动作的三段式在代码里对应哪些回调？（goal / feedback / result）

> ✅ 全部能回答 + 能默写代码，本课通过。